In [1]:
import os
import googlemaps
from datetime import datetime
import pandas as pd
import numpy as np
import random
import math

In [2]:
gmaps = googlemaps.Client(key=os.environ['GOOGLE_MAPS_API_KEY'])
locations = pd.read_csv('./csv/singapore_city_coordinates_improved.csv')
locations.drop(['bounding_box_1', 'bounding_box_2', 'bounding_box_3', 'bounding_box_4'], inplace=True, axis=1)
locations.dropna(subset=['latitude' , 'longitude'], inplace=True)
locations = locations[(locations.Place != 'Park') 
                      & (locations.Place != 'Trafalgar') 
                      & (locations.Place != 'Semakau')
                      & (locations.Place != 'Sudong')
                      & (locations.Place != 'Straits View')
                      & (locations.Place != 'City Terminals')
                      & (locations.Place != 'Murai')
                      & (locations.Place != 'Lim Chu Kang')
                      & (locations.Place != 'Pulau Seletar')
                      & (locations.Place != 'North-Eastern Islands')
                      & (locations.Place != 'Pulau Seletar')]
locations.reset_index(inplace=True, drop=True)
locations

,Place,City,Area,latitude,longitude
0,Bishan East,Bishan,central,1.351912,103.848971
1,Marymount,Bishan,central,1.348604,103.839393
2,Upper Thomson,Bishan,central,1.354639,103.832512
3,Alexandra Hill,Bukit Merah,central,1.277477,103.801880
4,Alexandra North,Bukit Merah,central,1.277477,103.801880
...,...,...,...,...,...
299,Seletar Hills,Serangoon,north-east,1.385464,103.865668
300,Serangoon Central,Serangoon,north-east,1.353514,103.871616
301,Serangoon Garden,Serangoon,north-east,1.362458,103.866013
302,Serangoon North,Serangoon,north-east,1.371338,103.872756


### Retrieving travel time from location to location by Google API

In [3]:
n=locations.shape[0]
distances_arr=np.empty([n, n])
for i_index in range(n):
    for j_index in range(n):
        print(i_index, j_index)
        distance = gmaps.distance_matrix(origins = (locations.iloc[i_index,-2], locations.iloc[i_index,-1]) , destinations=(locations.iloc[j_index,-2], locations.iloc[j_index,-1]), mode='driving', avoid='ferries')['rows'][0]['elements'][0]['duration']['text']       
        print(distance) 
        distances_arr[i_index,j_index]= distance

#Preventing Zero Time (Self-Time)
distances_arr[np.diag_indices_from(distances_arr)] = '99999'

0 0


TransportError: HTTPSConnectionPool(host='maps.googleapis.com', port=443): Max retries exceeded with url: /maps/api/distancematrix/json?avoid=ferries&destinations=1.3519117%2C103.8489708&mode=driving&origins=1.3519117%2C103.8489708&key=[REDACTED_API_KEY] (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1006)')))

In [ ]:
df_distances = pd.DataFrame(distances_arr, columns=list(locations['Place']))
df_distances['index_column'] = list(df_distances.columns)
df_distances.set_index('index_column', inplace=True)

# Export to CSV

In [ ]:
df_distances.to_csv('df_time.csv', index=True)  

# HQs to locations

In [ ]:
police_locations = pd.read_csv('./csv/policeheadquarters.csv')
police_locations.drop(['Address'], inplace=True, axis=1)
rows = police_locations.shape[0]
cols = locations.shape[0]
distances_arr=np.empty([rows, cols])
for i_index in range(rows):
    for j_index in range(cols):
        print(i_index, j_index)
        distance = int(gmaps.distance_matrix(origins = (police_locations.iloc[i_index,-2], police_locations.iloc[i_index,-1]) , destinations=(locations.iloc[j_index,-2], locations.iloc[j_index,-1]), mode='driving', avoid='ferries')['rows'][0]['elements'][0]['duration']['text'].split(' ')[0])     
        print(distance) 
        distances_arr[i_index,j_index]= distance

hq_to_locations = pd.DataFrame(distances_arr, columns=list(locations['Place']))
hq_to_locations['index_column'] = list(police_locations['Division'])
hq_to_locations.set_index('index_column', inplace=True)
hq_to_locations.to_csv('hq to locations.csv', index=True)      

0 0
9
0 1
12
0 2
15
0 3
24
0 4
24
0 5
26
0 6
24
0 7
24
0 8
24
0 9
25
0 10
27
0 11
26
0 12
23
0 13
27
0 14
28
0 15
25
0 16
23
0 17
25
0 18
25
0 19
24
0 20
18
0 21
17
0 22
21
0 23
19
0 24
20
0 25
23
0 26
24
0 27
23
0 28
20
0 29
22
0 30
19
0 31
20
0 32
22
0 33
25
0 34
18
0 35
25
0 36
24
0 37
24
0 38
16
0 39
18
0 40
13
0 41
18
0 42
14
0 43
14
0 44
13
0 45
18
0 46
12
0 47
17
0 48
16
0 49
11
0 50
16
0 51
20
0 52
24
0 53
21
0 54
21
0 55
21
0 56
24
0 57
20
0 58
18
0 59
19
0 60
17
0 61
21
0 62
17
0 63
17
0 64
15
0 65
14
0 66
17
0 67
15
0 68
16
0 69
16
0 70
15
0 71
15
0 72
22
0 73
20
0 74
23
0 75
23
0 76
27
0 77
23
0 78
27
0 79
20
0 80
25
0 81
23
0 82
21
0 83
25
0 84
19
0 85
20
0 86
26
0 87
26
0 88
27
0 89
26
0 90
30
0 91
19
0 92
27
0 93
22
0 94
22
0 95
21
0 96
20
0 97
20
0 98
19
0 99
18
0 100
17
0 101
19
0 102
14
0 103
20
0 104
21
0 105
17
0 106
18
0 107
15
0 108
22
0 109
24
0 110
23
0 111
21
0 112
29
0 113
24
0 114
21
0 115
18
0 116
19
0 117
14
0 118
16
0 119
15
0 120
16
0 121
14
0 122
13
0 12

PermissionError: [Errno 13] Permission denied: 'hq to locations.csv'

In [ ]:
hq_to_locations.to_csv('hq to locations.csv', index=True)      

In [ ]:
locations = pd.read_csv('./csv/df_time.csv', index_col=['index_column'])
random.seed(0)
n = len(locations.columns)
times = []
for i in range(n):
    if random.random() <= 0.05:
        times.append(random.randint(90, 120))
    else:
        times.append(random.randint(60, 90))


In [ ]:
df_times = pd.DataFrame(
    {'Places': locations.columns,
     'Amount of time to patrol': times
    })
df_times.index = df_times.index + 1
df_times.head()

,Places,Amount of time to patrol
1,Bishan East,84
2,Marymount,61
3,Upper Thomson,76
4,Alexandra Hill,89
5,Alexandra North,69


In [ ]:
df_times.to_csv('patrol.csv', index=True)  